# RooFit mini tutorial

RooFit is ROOT's toolkit for probability density functions (PDFs), datasets, and likelihood fits. This page introduces the objects used in the later examples; the complete comparisons are kept in the two analysis pages below.

Choose **Python / PyROOT** or **ROOT C++** below before reading the example. Both versions use the same model and random seed; the displayed fit result and figure are shared.



<div class="code-language-switch" role="group" aria-label="Code language">
  <span>Code language:</span>
  <button type="button" data-code-language="python" aria-pressed="true">Python / PyROOT</button>
  <button type="button" data-code-language="cpp" aria-pressed="false">ROOT C++</button>
</div>

<style>
.code-language-switch { display:none; gap:.5rem; align-items:center; margin:1rem 0; }
.code-language-switch button { padding:.3rem .8rem; border:1px solid #b8b8b8; border-radius:4px; background:#fff; cursor:pointer; }
.code-language-switch button[aria-pressed="true"] { color:#fff; background:#2f6f9f; border-color:#2f6f9f; }
.pyroot-code-marker { display:none; }
.pyroot-code-marker + .highlight {
  margin:.5rem 0 1rem;
  border:1px solid #d5d5d5;
  border-radius:2px;
  background:#f7f7f7;
}
.pyroot-code-marker + .highlight pre {
  margin:0;
  padding:.75rem 1rem;
  overflow-x:auto;
}
.pyroot-code-cell[hidden],
.jp-CodeCell .jp-Cell-inputWrapper[hidden] { display:none !important; }
</style>

<script>
document.addEventListener("DOMContentLoaded", function () {
  const buttons = document.querySelectorAll(".code-language-switch button");
  const pythonCells = Array.from(document.querySelectorAll(".pyroot-code-marker"))
    .map(function (marker) { return marker.closest(".jp-MarkdownCell"); })
    .filter(Boolean);
  pythonCells.forEach(function (cell) { cell.classList.add("pyroot-code-cell"); });
  const cppInputs = document.querySelectorAll(".jp-CodeCell .jp-Cell-inputWrapper");

  function selectLanguage(language) {
    pythonCells.forEach(function (cell) { cell.hidden = language !== "python"; });
    cppInputs.forEach(function (input) { input.hidden = language !== "cpp"; });
    buttons.forEach(function (button) {
      button.setAttribute("aria-pressed", String(button.dataset.codeLanguage === language));
    });
  }

  buttons.forEach(function (button) {
    button.addEventListener("click", function () { selectLanguage(button.dataset.codeLanguage); });
  });
  document.querySelector(".code-language-switch").style.display = "flex";
  selectLanguage("python");
});
</script>


## Core objects

- **RooRealVar** represents an observable or a fit parameter and defines its allowed range.
- **RooDataSet** stores event-by-event data for an unbinned likelihood.
- **RooDataHist** stores binned counts.
- **RooGaussian**, **RooExponential**, and other classes define normalized PDFs.
- **RooAddPdf** combines components; **RooExtendPdf** associates a PDF with an expected yield.
- **fitTo** performs the fit, and **RooPlot** displays the data and model.

## Likelihood terminology

- **unbinned likelihood** uses every measured event value directly;
- **binned likelihood**, also called a **Poisson likelihood** when each bin count is modeled as Poisson, uses the counts in fixed bins;
- **extended likelihood** includes the probability of the observed total event count and can therefore estimate yields as well as shape parameters.

The later examples state both the likelihood and the RooFit data object explicitly, so the statistical model is clear from the page itself.

## Normalization and ranges

RooFit normalizes a PDF over the range of its observable. The observable range is consequently part of the statistical model, not just an axis setting. In particular, <code>RooExponential</code> is proportional to $e^{a t}$; a decay requires $a<0$, and the corresponding lifetime is $T=-1/a$.

## ROOT Fit and RooFit

<code>TH1::Fit</code>, <code>TGraph::Fit</code>, and <code>TF1</code> are direct tools for common histogram and graph fits. RooFit is useful when the analysis needs normalized PDFs, unbinned data, component models, explicit yields, or extended likelihood. A simple histogram fit does not automatically need RooFit.

## Minimal example

The example below generates an unbinned Gaussian sample, fits its mean and width, prints the fit status and parameter errors, and draws the fitted PDF. The same model is implemented in PyROOT and ROOT C++.

Further examples:

- [Exponential decay: unbinned and binned likelihood](likelihood_decay.html)
- [Signal plus background: unbinned, binned, and extended likelihood](likelihood_signal_background.html)

Official references: [RooFit tutorials](https://root.cern/doc/master/group__tutorial__roofit.html) and [RooFit manual](https://root.cern/manual/roofit/).


<div class="pyroot-code-marker"></div>

```python
import ROOT

ROOT.RooRandom.randomGenerator().SetSeed(42)

# Observable and PDF parameters
x = ROOT.RooRealVar("x", "x", -5, 5)
mean = ROOT.RooRealVar("mean", "mean", 0.5, -2, 2)
sigma = ROOT.RooRealVar("sigma", "sigma", 1.5, 0.1, 3)
model = ROOT.RooGaussian("model_py", "Gaussian PDF", x, mean, sigma)

# Generate event-by-event data and fit the normalized PDF.
data = model.generate(ROOT.RooArgSet(x), 1000)
result = model.fitTo(
    data,
    ROOT.RooFit.Save(True),
    ROOT.RooFit.PrintLevel(-1),
)

print(f"status = {result.status()}, covQual = {result.covQual()}")
print(f"mean  = {mean.getVal():.4f} +/- {mean.getError():.4f}")
print(f"sigma = {sigma.getVal():.4f} +/- {sigma.getError():.4f}")

# Binning is used only to display the unbinned data.
frame = x.frame(ROOT.RooFit.Title("Unbinned Gaussian fit"))
data.plotOn(
    frame,
    ROOT.RooFit.Binning(30),
    ROOT.RooFit.DataError(ROOT.RooAbsData.Poisson),
)
model.plotOn(frame, ROOT.RooFit.LineColor(ROOT.kRed))

canvas = ROOT.TCanvas("c_roofit_intro_py", "RooFit", 700, 500)
frame.Draw()
canvas.Draw()
```


In [1]:
#include "RooRealVar.h"
#include "RooGaussian.h"
#include "RooDataSet.h"
#include "RooPlot.h"
#include "RooFitResult.h"
#include "RooRandom.h"
#include "TCanvas.h"
#include <iostream>

using namespace RooFit;

RooRandom::randomGenerator()->SetSeed(42);

// Observable and PDF parameters
RooRealVar x("x", "x", -5, 5);
RooRealVar mean("mean", "mean", 0.5, -2, 2);
RooRealVar sigma("sigma", "sigma", 1.5, 0.1, 3);
RooGaussian model("model", "Gaussian PDF", x, mean, sigma);

// Generate event-by-event data and fit the normalized PDF.
auto dataSet = model.generate(x, 1000);
auto result = model.fitTo(*dataSet, Save(true), PrintLevel(-1));

std::cout << "status = " << result->status()
          << ", covQual = " << result->covQual() << "\n";
std::cout << "mean  = " << mean.getVal() << " +/- " << mean.getError() << "\n";
std::cout << "sigma = " << sigma.getVal() << " +/- " << sigma.getError() << std::endl;

// Binning is used only to display the unbinned data.
auto frame = x.frame(Title("Unbinned Gaussian fit"));
dataSet->plotOn(frame, Binning(30), DataError(RooAbsData::Poisson));
model.plotOn(frame, LineColor(kRed));

auto canvas = new TCanvas("c_roofit_intro", "RooFit", 700, 500);
frame->Draw();
canvas->Draw();

[#1] INFO:Fitting -- RooAbsPdf::fitTo(model_over_model_Int[x]) fixing normalization set for coefficient determination to observables in data
[#1] INFO:Fitting -- using generic CPU library compiled with no vectorizations
[#1] INFO:Fitting -- Creation of NLL object took 14.9274 ms
[#1] INFO:Fitting -- RooAddition::defaultErrorLevel(nll_model_over_model_Int[x]_modelData) Summation contains a RooNLLVar, using its error level
[#1] INFO:Minimization -- [fitFCN] No discrete parameters, performing continuous minimization only
status = 0, covQual = 3
mean  = 0.500853 +/- 0.0478072
sigma = 1.4999 +/- 0.0349658
